In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
# dataloader_binary.py 먼저 만들기
# model_binary_resnet.py로 모델 구조
# train_binary.py 작성해서 학습
# 성능 확인 후 → 필요 시 증강 적용 또는 ResNet34/Dropout 확장
# 사용 모델 RetNet18 모델

In [ ]:
# RetNet18 모델 평가 / 혼동행렬, 정확도 그래프 시각화
"""
모델 로드 (best_model.pth)
테스트 데이터 평가 : 정확도,정밀도,재현율 등 계산
혼동행렬 시각화 : 어떤 클래스에서 오류나는지 체크
예측 확률, 오답 이미지 보기 : 잘못 분류한 이미지 확인 가능

"""

In [2]:
import torch 
import toch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoder

from model_binary_resnet import get_resnet18_binary_model
import os
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# 경로 설정
data_dir = "" # ★★★★★★★★★★★★ 경로 설정 - 수정 필요
model_path = "best 모델 이름" # ★★★★ 파일명 수정 필요

# 테스트용 transform
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], # 평균(mean) red:0.485 / green:0.456 / blue;0.406
                         [0.229, 0.224, 0.225]) # 표준편차 (std) red:0.229 / green:0.224 / blue;0.225
# RGB 각 채널의 ImageNet 평균/표준편차로 정규화  , ImageNet 기준 pretrained=True 모델 사용시 mean/std 반드시 사용해야함 
])

# 테스트 데이터 로딩
test_dataset = datasets.ImageFolder(os.path.join(data_dir,'test'), transform=transform)
test_loader = DataLoder(test_dataset, batch_size=32, shuffle=False)

# 클래스 이름 확인
class_naems = test_dataset.classes # food, not_food
print("클래스 라벨 :", class_naems)

# 모델 불러오기 
model = get_resnet18_binary_model(pretrained=False).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# 예측 수행
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images,labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        
        apll_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 평가 지표 출력
print("\n classification report")
print(classification_report(all_labels, all_preds, target_names=class_names))

# 혼동 행렬 출력
cm = confusion_metrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_metrix=cm, display_labels=class_names)

plt.figure(figsize=(5,5))
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix (food vs not_food)")
plt.savefig("confusion_matrix_binary.png")  # 이미지로 저장
plt.show()